# UA-FCCL: sensitivity to lambda and p (Colab)
Run the cells **one by one, from top to bottom**. If this Colab session was used before, first click **Runtime > Restart session**.

**Files needed on your computer:** `ua_gate.py`, `sens.py` (both sent in chat), and `Shill Bidding Dataset.csv`.

The notebook recomputes the method for lambda in {0.25, 0.5, 0.75} and p in {1.5, 2, 3} on all nine datasets (20 splits each). The main runs used lambda = 0.5 and p = 2. Total run time is about 15 to 25 minutes.

## Cell 1: get the repo, the clustered-conformal code and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl class-conditional-conformal
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
!git clone -q https://github.com/tiffanyding/class-conditional-conformal.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
import os
os.environ['CCL_DING_DIR'] = '/content/class-conditional-conformal'
!ls ccl_helpers.py

## Cell 2: upload `ua_gate.py` and `sens.py`
Click **Choose Files** and select both files together.

In [ ]:
import glob, os, shutil
from google.colab import files
up = files.upload()
for base in ['ua_gate', 'sens']:
    c = sorted(glob.glob(base + '*.py'), key=os.path.getmtime)
    if c[-1] != base + '.py': shutil.copy(c[-1], base + '.py')
    print('using', c[-1])
!ls ua_gate.py sens.py

## Cell 3: download the datasets (ignore the Shill Bidding message)

In [ ]:
!python 01_prepare_data.py
!ls data

## Cell 4: upload `Shill Bidding Dataset.csv`

In [ ]:
import shutil
from google.colab import files
up = files.upload()
for fn in up:
    shutil.move(fn, 'data/Shill Bidding Dataset.csv')
!ls data

## Cell 5: load the nine datasets

In [ ]:
import numpy as np, pandas as pd, time, os, warnings
warnings.filterwarnings('ignore')
import ccl_helpers as H
from sens import run_sens, summarise
files_map = {'shill': 'Shill Bidding Dataset.csv', 'wilt': 'wilt.csv', 'htru2': 'htru2.csv', 'magic': 'magic.csv',
             'pageblocks': 'pageblocks.csv', 'drybean': 'drybean.csv', 'shuttle': 'shuttle.csv',
             'yeast': 'yeast.csv', 'ecoli': 'ecoli.csv'}
data = {n: H.load('data/' + f) for n, f in files_map.items()}
for n, (X, y) in data.items():
    print(n, X.shape, 'class sizes:', np.bincount(y).tolist())

## Cell 6: quick test on Ecoli (about 1 minute). Check that a table prints.

In [ ]:
t0 = time.time()
X, y = data['ecoli']
d = run_sens('ecoli', X, y, range(3))
print('time', round(time.time() - t0, 1), 's')
print(summarise(d).to_string())

## Cell 7: run all nine datasets (results are saved after each one)

In [ ]:
for n in files_map:
    f = f'sens_{n}.csv'
    if os.path.exists(f):
        print(n, 'already done, skipping'); continue
    t0 = time.time()
    X, y = data[n]
    run_sens(n, X, y, range(20)).to_csv(f, index=False)
    print(n, 'done in', round((time.time() - t0) / 60, 1), 'min')

## Cell 8: summary
Columns: `coverage` (all classes) and `rare_cov` (rarest class) of the adaptive sets at alpha = 0.1; `agree` is the share of test cases where the nearest pseudo-concept equals the forest decision; `gate_share` is the share of cases with an explanation shown at delta = 0.10; `over` is the share of splits where the unfaithful rate among shown cases was above delta.

In [ ]:
S = summarise(pd.concat([pd.read_csv(f'sens_{n}.csv') for n in files_map]))
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 2000)
S.to_csv('sens_summary.csv')
print(S.to_string())

## Cell 9: download everything and send it to me

In [ ]:
!zip -q sens_results.zip sens_*.csv
from google.colab import files
files.download('sens_results.zip')